# Chg2Cap: Colab training and evaluation

This notebook clones a selected GitHub ref, prepares LEVIR-CC, trains Chg2Cap and evaluates the test split. Code and images stay under `/content`; checkpoints, logs, environment records and evaluation JSON files are written to Google Drive.

Select a GPU runtime and run cells in order. GPU availability and runtime versions are checked rather than assuming an A100 or a specific Python version. This notebook has been structurally validated locally; execution on Colab has not yet been verified.

The selected GitHub ref must contain the changes accompanying this notebook. Local edits are not transferred by cloning GitHub. Commit and push them to your fork yourself before using this notebook.

## 1. Configuration

Set a branch, tag or full commit SHA in `REPO_REF`. Use a commit SHA for a repeatable code snapshot. The default branch below is a proposed branch and must first be published to the fork.

The dataset URL comes from the supplied RSICC notebook. Its availability and contents have not been checked in this session. Alternatively, set `DATASET_SOURCE` to a ZIP or dataset folder on your mounted Drive; it takes precedence over the URL.

Use a unique `RUN_NAME` for each experiment. To resume an interrupted run, reuse its name and set `RESUME=True`. `NUM_EPOCHS` is the total target epoch count, including completed epochs. Checkpoints are written at epoch boundaries, not every batch.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import subprocess
import shutil
import zipfile

REPO_URL = "https://github.com/yakupgulcan/Chg2Cap.git"
REPO_REF = "feature/modernize-runtime"  # Branch, tag or commit SHA
PROJECT_DIR = Path("/content/Chg2Cap")
DATASET_URL = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"
DATASET_SOURCE = ""  # e.g. /content/drive/MyDrive/datasets/LEVIR_CC.zip
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/Chg2Cap_Outputs")
RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
NUM_EPOCHS = 20
TRAIN_BATCHSIZE = 2  # Adjust after verifying the assigned GPU's memory
WORKERS = 2
RESUME = False
RUN_SMOKE_CHECK = True

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is required by this notebook.")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be one directory name.")

## 2. Mount Drive and clone the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} exists. Use a fresh runtime or choose a new project directory.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
for required in ["requirements-colab.txt", "runtime.py", "scripts/prepare_meteor.py", "scripts/smoke_check.py"]:
    if not (PROJECT_DIR / required).is_file():
        raise FileNotFoundError(f"Selected Git ref does not contain {required}. Publish the prepared changes to your fork first.")
RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_NAME
if RUN_DIR.exists() and not RESUME:
    raise FileExistsError("Run already exists. Choose a new RUN_NAME or set RESUME=True.")
if RESUME and not (RUN_DIR / "checkpoints/last.pth").is_file():
    raise FileNotFoundError("Resume requested but checkpoints/last.pth is missing.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
RESULTS_DIR = RUN_DIR / "test_results"
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. Dependencies and runtime record

Keep Colab's matched PyTorch, torchvision and CUDA build. Install only the additional requirements in a subprocess; the notebook does not import PyTorch before installation. A changed runtime must pass the checks below. Restart the runtime if packages were already imported before this cell.

Java is required for METEOR. Its resource file is downloaded with SHA-256 verification. The model uses a CUDA GPU; CPU training is not supported by this implementation.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "gdown==5.2.0"], check=True)
if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-17-jre-headless"], check=True)
subprocess.run([sys.executable, "scripts/prepare_meteor.py"], check=True)

# Check the actual packages used by train.py, not cached notebook imports.
check = subprocess.run([sys.executable, "-c", "import torch, torchvision; assert torch.cuda.is_available(), 'Select a GPU runtime'; x=torch.ones(4, device='cuda'); assert (x+x).sum().item()==8; from eval_func.meteor.meteor import Meteor; m=Meteor(); print(m.compute_score([['1 2 3']], [['1 2 3']])[0]); m.close()"], check=True)
from runtime import environment_info, write_json
record_name = datetime.now(timezone.utc).strftime("environment_%Y%m%d_%H%M%S")
write_json(RUN_DIR / (record_name + ".json"), environment_info())
packages = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / (record_name + "_packages.txt")).write_text(packages)
package_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(RUN_DIR / (record_name + "_pip_check.txt")).write_text(package_check.stdout + package_check.stderr)
print(package_check.stdout)
if package_check.returncode:
    raise RuntimeError("Resolve the recorded dependency conflicts before training.")
print(json.dumps(environment_info(), indent=2))

## 4. Prepare LEVIR-CC locally

Chg2Cap reads paired images and token text files, not RSICC's HDF5 files. The source must contain `LevirCCcaptions.json` and `images/{train,val,test}/{A,B}`. ZIP archives with one enclosing folder are supported. Images are copied to runtime storage; Google Drive holds outputs.

Preprocessing verifies caption image paths and rewrites split lists without duplicating rows. Vocabulary uses threshold 5, matching the repository recommendation.

In [ ]:
DATASET_ROOT = PROJECT_DIR / "datasets/LEVIR_CC"
if not DATASET_ROOT.exists():
    if DATASET_SOURCE:
        source = Path(DATASET_SOURCE)
        if not source.exists():
            raise FileNotFoundError(source)
    else:
        if not DATASET_URL:
            raise ValueError("Set DATASET_SOURCE or DATASET_URL.")
        import gdown
        source = Path("/content/LEVIR_CC_download.zip")
        if gdown.download(DATASET_URL, str(source), fuzzy=True) is None:
            raise RuntimeError("Dataset download failed.")
    if source.is_file():
        staging = Path("/content/Chg2Cap_dataset_extract")
        if staging.exists():
            raise FileExistsError("Dataset extraction staging exists; use a fresh runtime.")
        staging.mkdir()
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                target = (staging / member.filename).resolve()
                if not target.is_relative_to(staging.resolve()):
                    raise ValueError("Archive member escapes the extraction directory.")
            archive.extractall(staging)
        candidates = list(staging.rglob("LevirCCcaptions.json"))
    else:
        candidates = list(source.rglob("LevirCCcaptions.json"))
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one captions JSON, found {len(candidates)}.")
    dataset_folder = candidates[0].parent
    if not (dataset_folder / "images").is_dir():
        raise FileNotFoundError("images directory must be next to LevirCCcaptions.json")
    DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(dataset_folder, DATASET_ROOT)

PREPARED_DIR = PROJECT_DIR / "data/LEVIR_CC"
subprocess.run([sys.executable, "preprocess_data.py", "--dataset", "LEVIR_CC", "--dataset_root", str(DATASET_ROOT), "--word_count_threshold", "5"], check=True)
manifest = json.loads((PREPARED_DIR / "manifest.json").read_text())
if manifest["max_length"] > 41:
    raise ValueError("Captions exceed the default max_length=41; configure train and test consistently.")
archive_dir = RUN_DIR / "prepared"
if RESUME:
    for name in ["train.txt", "val.txt", "test.txt", "vocab.json", "manifest.json"]:
        if (archive_dir / name).read_bytes() != (PREPARED_DIR / name).read_bytes():
            raise ValueError(f"Prepared {name} differs from the saved run.")
else:
    archive_dir.mkdir()
    for name in ["train.txt", "val.txt", "test.txt", "vocab.json", "manifest.json"]:
        shutil.copy2(PREPARED_DIR / name, archive_dir / name)
print(manifest)

## 5. Smoke check

Before a full run, test two training batches, four validation pairs, epoch resume and four test pairs. This checks checkpoint loading and metrics on the assigned runtime. It does not establish final model quality or full dataset performance. Smoke checkpoints are separate from the experiment checkpoints.

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + json.dumps(command) + "\n")
        log.flush()
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            return_code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

if RUN_SMOKE_CHECK:
    smoke_dir = RUN_DIR / ("smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
    run_logged([sys.executable, "-u", "scripts/smoke_check.py", "--dataset_root", str(DATASET_ROOT), "--output_dir", str(smoke_dir), "--resume_check"], RUN_DIR / "smoke.log")

## 6. Train

`last.pth` stores model, optimizer and scheduler states at each completed training epoch, before validation, and is updated after validation. `best.pth` stores inference weights selected by validation BLEU-4. If validation is interrupted, resuming `last.pth` starts at the next epoch; the interrupted epoch's validation is not replayed. Within-epoch progress and random generator states are not restored.

Use a new run directory for fresh training. Reduce batch size if CUDA runs out of memory. Logs and epoch checkpoints are written directly to Drive; atomic replacement is attempted, but Drive mount durability during disconnection has not been tested.

In [ ]:
command = [sys.executable, "-u", "train.py", "--data_folder", str(DATASET_ROOT / "images"), "--savepath", str(CHECKPOINT_DIR), "--num_epochs", str(NUM_EPOCHS), "--train_batchsize", str(TRAIN_BATCHSIZE), "--workers", str(WORKERS), "--print_freq", "100"]
if RESUME:
    command += ["--checkpoint", str(CHECKPOINT_DIR / "last.pth")]
write_json(RUN_DIR / "notebook_config.json", {"repo_url": REPO_URL, "repo_ref": REPO_REF, "dataset_url": DATASET_URL, "dataset_source": DATASET_SOURCE, "run_name": RUN_NAME, "epochs": NUM_EPOCHS, "batchsize": TRAIN_BATCHSIZE, "workers": WORKERS, "resume": RESUME})
run_logged(command, RUN_DIR / "train.log")

## 7. Evaluate the test split

Select the validation-selected `best.pth`; do not use test scores to select a checkpoint. The evaluator processes one image pair per batch and saves overall, change and no-change metrics, predictions, configuration and environment records. Re-running evaluation with the same results directory replaces its JSON outputs; the log is appended.

In [ ]:
EVAL_CHECKPOINT = "best.pth"
if not (CHECKPOINT_DIR / EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(CHECKPOINT_DIR / EVAL_CHECKPOINT)
run_logged([sys.executable, "-u", "test.py", "--data_folder", str(DATASET_ROOT / "images"), "--savepath", str(CHECKPOINT_DIR), "--checkpoint", EVAL_CHECKPOINT, "--workers", str(WORKERS), "--results_dir", str(RESULTS_DIR)], RUN_DIR / "test.log")
print(json.dumps(json.loads((RESULTS_DIR / "metrics.json").read_text()), indent=2))